# Simulador de Base de Alunos

> **Portfolio pessoal.** Este notebook foi extraído de um projeto real de automação de processos acadêmicos e passou por um processo de anonimização: nomes de instituições, pessoas e caminhos locais foram substituídos por termos genéricos, e todos os outputs de execução (que continham dados reais de alunos/professores) foram removidos. O código e a lógica de negócio foram mantidos como estavam em produção.

Gera uma base de alunos sintética para testar e demonstrar os notebooks acima sem depender de dados reais.

---


# Simulador de Base de Alunos

In [ ]:
import pandas as pd
import numpy as np
import re
import logging
import joblib
import pandera as pa
import mlflow
import warnings
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, silhouette_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
import lightgbm as lgb
import optuna

# =====================================================================
# 1. LOGGING E CONFIGURAÇÃO
# =====================================================================
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING) # Silencia logs spam do Optuna
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

BASE_DIR = Path('.')

CONFIG = {
    'metas_futuras': {
        '5226': {'meta_bruta': 64939, 'efetividade': 0.92},
        '5326': {'meta_bruta': 97277, 'efetividade': 0.88},
        '5426': {'meta_bruta': 58366, 'efetividade': 0.86},
    },
    'optuna_trials': 100, # <-- Quantidade de vezes que o modelo vai iterar e tentar melhorar
    'artefato_modelo': 'modelo_lgbm_otimizado.pkl',
    'arquivo_saida': 'PROJECAO_BASE_FUTURA_FINAL.csv'
}

# =====================================================================
# 2. CONTRATOS DE DADOS (Pandera Schemas)
# =====================================================================
schema_base = pa.DataFrameSchema({
    "COD_CURSO": pa.Column(str, coerce=True),
    "Qtd_Ativos": pa.Column(int, pa.Check(lambda x: x >= 0), coerce=True)
})

schema_captacao = pa.DataFrameSchema({
    "Curso - Cod.": pa.Column(str, coerce=True),
    "Mat. - Real": pa.Column(int, pa.Check(lambda x: x >= 0), coerce=True)
})

# =====================================================================
# 3. ETAPA DE DESCOBERTA CRONOLÓGICA
# =====================================================================
def extrair_modulo(nome_arquivo):
    match = re.search(r'(\d{4})', str(nome_arquivo))
    return match.group(1) if match else None

logger.info("Varrendo diretório e construindo linha do tempo das bases...")
arquivos_base = list(BASE_DIR.glob('basealunos*.xlsx'))

modulos_extraidos = [extrair_modulo(f.name) for f in arquivos_base if extrair_modulo(f.name)]
modulos_historicos = sorted(modulos_extraidos, key=lambda x: x[-2:] + x[:2])
logger.info(f"Módulos históricos em ordem cronológica real: {modulos_historicos}")

# =====================================================================
# 4. EXTRAÇÃO DE TAXAS E CLUSTERIZAÇÃO DINÂMICA
# =====================================================================
def processar_motor_evasao(base_dir):
    arquivos_ren = list(base_dir.glob('renovacao*.xlsx'))
    if not arquivos_ren: return pd.DataFrame()
    
    df_hist = pd.concat((pd.read_excel(arq) for arq in tqdm(arquivos_ren, desc="Lendo Renovação")), ignore_index=True).fillna(0)
    
    df_hist['Potencial'] = df_hist[['Aptos', 'Matriculados', 'Ingressantes', 'Winback']].sum(axis=1)
    df_hist['Evasao'] = df_hist[['Evasão Veteranos', 'Evasão Ingressantes', 'Evasão Winback', 'Não Matriculados Calouros']].sum(axis=1)
    df_hist['Concluintes'] = df_hist[['Possível Formando', 'Concluídos']].sum(axis=1)
    
    df_taxas = df_hist.groupby('COD. CURSO', as_index=False)[['Potencial', 'Evasao', 'Concluintes']].sum()
    df_taxas['Taxa_Ev'] = np.where(df_taxas['Potencial'] > 0, df_taxas['Evasao'] / df_taxas['Potencial'], 0)
    df_taxas['Taxa_Co'] = np.where(df_taxas['Potencial'] > 0, df_taxas['Concluintes'] / df_taxas['Potencial'], 0)
    
    X_scaled = StandardScaler().fit_transform(df_taxas[['Taxa_Ev', 'Taxa_Co']])
    melhor_k, melhor_score = 3, -1
    
    for k in range(2, min(6, len(df_taxas))):
        kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans_temp.fit_predict(X_scaled)
        if silhouette_score(X_scaled, labels) > melhor_score:
            melhor_score = silhouette_score(X_scaled, labels)
            melhor_k = k
            
    kmeans_final = KMeans(n_clusters=melhor_k, random_state=42, n_init=10)
    df_taxas['Cluster_Risco'] = kmeans_final.fit_predict(X_scaled)
    
    return df_taxas.rename(columns={'COD. CURSO': 'Codigo_Curso'})

def calcular_coeficientes_captacao(base_dir):
    arquivos_cap = list(base_dir.glob('captacao*.xlsx'))
    if not arquivos_cap: return pd.DataFrame()
    
    dfs = [schema_captacao.validate(pd.read_excel(arq)) for arq in tqdm(arquivos_cap, desc="Validando Captações")]
    df_cap = pd.concat(dfs, ignore_index=True).groupby('Curso - Cod.', as_index=False)['Mat. - Real'].sum()
    df_cap['Coef_Global'] = df_cap['Mat. - Real'] / df_cap['Mat. - Real'].sum()
    return df_cap.rename(columns={'Curso - Cod.': 'Codigo_Curso'})

df_taxas_globais = processar_motor_evasao(BASE_DIR)
df_coef_captacao = calcular_coeficientes_captacao(BASE_DIR)

# =====================================================================
# 5. CONSTRUÇÃO DO DATASET DE TREINO (Com Sazonalidade por Módulo)
# =====================================================================
registros_treino = []

for i in tqdm(range(len(modulos_historicos) - 1), desc="Construindo Base Treino"):
    mod_atual, mod_futuro = modulos_historicos[i], modulos_historicos[i+1]
    
    arq_base_ant = BASE_DIR / f'basealunos{mod_atual}.xlsx'
    arq_cap = BASE_DIR / f'captacao{mod_futuro}.xlsx'
    arq_base_pos = BASE_DIR / f'basealunos{mod_futuro}.xlsx'
    
    if not (arq_base_ant.exists() and arq_base_pos.exists()): continue
        
    df_b_ant = schema_base.validate(pd.read_excel(arq_base_ant)).groupby('COD_CURSO', as_index=False)['Qtd_Ativos'].sum()
    df_b_pos = schema_base.validate(pd.read_excel(arq_base_pos)).groupby('COD_CURSO', as_index=False)['Qtd_Ativos'].sum()
    
    df_c = schema_captacao.validate(pd.read_excel(arq_cap)).groupby('Curso - Cod.', as_index=False)['Mat. - Real'].sum() if arq_cap.exists() else pd.DataFrame(columns=['Curso - Cod.', 'Mat. - Real'])
    
    df_b_ant.columns, df_b_pos.columns, df_c.columns = ['Codigo_Curso', 'Base_Anterior'], ['Codigo_Curso', 'Base_Futura'], ['Codigo_Curso', 'Cap_Real']
    
    df_transicao = df_b_ant.merge(df_c, on='Codigo_Curso', how='outer').fillna(0)
    df_transicao = df_transicao.merge(df_b_pos, on='Codigo_Curso', how='left').fillna(0)
    df_transicao = df_transicao.merge(df_taxas_globais[['Codigo_Curso', 'Taxa_Ev', 'Taxa_Co', 'Cluster_Risco']], on='Codigo_Curso', how='left')
    
    df_transicao['Mes_Modulo'] = int(str(mod_futuro)[:2])
    df_transicao['Is_Pico_Captacao'] = np.where(df_transicao['Mes_Modulo'] == 51, 1, 0)
    
    df_transicao['Base_Bruta'] = df_transicao['Base_Anterior'] + df_transicao['Cap_Real']
    df_transicao['Saidas_Reais'] = np.clip(df_transicao['Base_Bruta'] - df_transicao['Base_Futura'], 0, None)
    
    df_transicao['Target_Taxa_Perda'] = np.where(df_transicao['Base_Bruta'] > 0, df_transicao['Saidas_Reais'] / df_transicao['Base_Bruta'], 0)
    df_transicao = df_transicao[df_transicao['Base_Bruta'] > 0]
    
    registros_treino.append(df_transicao)

df_treino = pd.concat(registros_treino, ignore_index=True).fillna(0)
df_treino['Cluster_Risco'] = df_treino['Cluster_Risco'].astype('category')
df_treino['Mes_Modulo'] = df_treino['Mes_Modulo'].astype('category')

# =====================================================================
# 6. MODELAGEM AUTOESCALÁVEL (Optuna + LightGBM Dinâmico)
# =====================================================================
from sklearn.model_selection import KFold, TimeSeriesSplit

FEATURES = ['Base_Anterior', 'Cap_Real', 'Taxa_Ev', 'Taxa_Co', 'Cluster_Risco', 'Mes_Modulo', 'Is_Pico_Captacao']
X = df_treino[FEATURES]
y = df_treino['Target_Taxa_Perda']

# --- INTELIGÊNCIA DE ESCALA ---
# O código descobre o tamanho da sua base lendo a quantidade de módulos disponíveis
qtd_modulos_historicos = len(modulos_historicos)
base_pequena = qtd_modulos_historicos <= 5

if base_pequena:
    logger.info(f"Modo Conservador Ativado: Foram detectados apenas {qtd_modulos_historicos} módulos históricos.")
    limites_optuna = {
        'n_estimators': (50, 150),
        'max_depth': (2, 5),
        'learning_rate': (0.01, 0.05),
        'min_child_samples': (5, 20)
    }
    # KFold é melhor para embaralhar os poucos dados e dar uma nota justa
    validador_cv = KFold(n_splits=min(5, len(X)//10), shuffle=True, random_state=42)
else:
    logger.info(f"Modo Agressivo Ativado: Base rica com {qtd_modulos_historicos} módulos históricos detectados.")
    limites_optuna = {
        'n_estimators': (100, 800),
        'max_depth': (3, 9),
        'learning_rate': (0.01, 0.1),
        'min_child_samples': (10, 40)
    }
    # Quando há muito histórico, o TimeSeriesSplit volta a ser a melhor opção para respeitar a cronologia
    validador_cv = TimeSeriesSplit(n_splits=min(5, qtd_modulos_historicos - 2))
# ------------------------------

def objective(trial):
    """Função do Optuna que ajusta os limites conforme o tamanho da base"""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', *limites_optuna['n_estimators']),
        'learning_rate': trial.suggest_float('learning_rate', limites_optuna['learning_rate'][0], limites_optuna['learning_rate'][1], log=True),
        'max_depth': trial.suggest_int('max_depth', *limites_optuna['max_depth']),
        'num_leaves': trial.suggest_int('num_leaves', 5, 30),
        'min_child_samples': trial.suggest_int('min_child_samples', *limites_optuna['min_child_samples']),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 2.0),
        'random_state': 42,
        'objective': 'regression',
        'verbose': -1
    }
    
    modelo_teste = lgb.LGBMRegressor(**params)
    score = cross_val_score(modelo_teste, X, y, cv=validador_cv, scoring='r2').mean()
    return score

logger.info(f"Iniciando Optuna (Testando {CONFIG['optuna_trials']} combinações com limites dinâmicos)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=CONFIG['optuna_trials'], show_progress_bar=True)

melhores_parametros = study.best_params
melhores_parametros.update({'random_state': 42, 'objective': 'regression', 'verbose': -1})

logger.info(f"Otimização concluída! Melhor R² Encontrado: {study.best_value*100:.2f}%")

# Treina o modelo definitivo
modelo_lgb_final = lgb.LGBMRegressor(**melhores_parametros)

mlflow.set_experiment("Simulador_Evasao_Alunos")
with mlflow.start_run():
    mlflow.log_params(melhores_parametros)
    mlflow.log_metric("r2_cv_best", study.best_value)
    
    modelo_lgb_final.fit(X, y)
    r2_treino = r2_score(y, modelo_lgb_final.predict(X))
    
    mlflow.log_metric("r2_treino_final", r2_treino)
    mlflow.lightgbm.log_model(modelo_lgb_final, "modelo_lgbm")

caminho_modelo = BASE_DIR / CONFIG['artefato_modelo']
joblib.dump(modelo_lgb_final, caminho_modelo)

# =====================================================================
# 7. PROJEÇÃO EM CADEIA
# =====================================================================
ultimo_modulo = modulos_historicos[-1]
df_projecao = pd.read_excel(BASE_DIR / f'basealunos{ultimo_modulo}.xlsx').groupby(['COD_CURSO', 'CURSO_NOME'], as_index=False)['Qtd_Ativos'].sum()
df_projecao.columns = ['Codigo_Curso', 'CURSO_NOME', 'Base_Atual']

df_projecao = df_projecao.merge(df_taxas_globais[['Codigo_Curso', 'Taxa_Ev', 'Taxa_Co', 'Cluster_Risco']], on='Codigo_Curso', how='left').fillna(0)
df_projecao = df_projecao.merge(df_coef_captacao, on='Codigo_Curso', how='left').fillna(0)
df_projecao['Cluster_Risco'] = df_projecao['Cluster_Risco'].astype('category')

total_coef_ativo = df_projecao['Coef_Global'].sum()
df_projecao['Coef_Global_Ajustado'] = np.where(total_coef_ativo > 0, df_projecao['Coef_Global'] / total_coef_ativo, 0)

logger.info(f"Identificados {len(df_projecao)} cursos ativos. Iniciando simulações longitudinais...")

for mod_futuro, config in tqdm(CONFIG['metas_futuras'].items(), desc="Simulando Projeções"):
    meta_liquida = config['meta_bruta'] * config['efetividade']
    col_cap = f'Cap_Prev_{mod_futuro}'
    
    df_projecao[col_cap] = (df_projecao['Coef_Global_Ajustado'] * meta_liquida).round(0)
    
    df_projecao['Mes_Modulo'] = int(str(mod_futuro)[:2])
    df_projecao['Is_Pico_Captacao'] = np.where(df_projecao['Mes_Modulo'] == 51, 1, 0)
    df_projecao['Mes_Modulo'] = df_projecao['Mes_Modulo'].astype('category')
    
    X_futuro = df_projecao[['Base_Atual', col_cap, 'Taxa_Ev', 'Taxa_Co', 'Cluster_Risco', 'Mes_Modulo', 'Is_Pico_Captacao']].copy()
    X_futuro.columns = FEATURES 
    
    taxa_perda_prevista = np.clip(modelo_lgb_final.predict(X_futuro), 0, 1)
    
    base_bruta = df_projecao['Base_Atual'] + df_projecao[col_cap]
    saidas_ajustadas = (base_bruta * taxa_perda_prevista).round(0)
    
    peso_ev = np.where((df_projecao['Taxa_Ev'] + df_projecao['Taxa_Co']) == 0, 0.8, 
                       df_projecao['Taxa_Ev'] / (df_projecao['Taxa_Ev'] + df_projecao['Taxa_Co']))
    
    col_ev = f'Ev_Prev_{mod_futuro}'
    col_fo = f'Form_Prev_{mod_futuro}'
    col_base = f'Base_Final_{mod_futuro}'
    
    df_projecao[col_ev] = (saidas_ajustadas * peso_ev).round(0)
    df_projecao[col_fo] = saidas_ajustadas - df_projecao[col_ev]
    df_projecao[col_base] = base_bruta - saidas_ajustadas
    
    df_projecao['Base_Atual'] = df_projecao[col_base]
    
    # Exibe os totais detalhados do módulo
    logger.info(
        f"➜ Módulo {mod_futuro} | "
        f"Captação (+): {df_projecao[col_cap].sum():,.0f} | "
        f"Evasão (-): {df_projecao[col_ev].sum():,.0f} | "
        f"Formandos (-): {df_projecao[col_fo].sum():,.0f} | "
        f"Base Final (=): {df_projecao[col_base].sum():,.0f}"
    )

df_projecao.drop(columns=['Coef_Global', 'Coef_Global_Ajustado', 'Base_Atual', 'Mes_Modulo', 'Is_Pico_Captacao'], inplace=True)
arquivo_saida = BASE_DIR / CONFIG['arquivo_saida']
df_projecao.to_csv(arquivo_saida, sep=';', decimal=',', index=False, encoding='utf-8-sig')

logger.info(f"Pipeline completo finalizado! Arquivo exportado: {arquivo_saida.name}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. Configuração do estilo visual (fundo limpo, sem grades pesadas)
sns.set_theme(style="whitegrid")

# 2. Coleta dos dados que já estão na memória do seu Notebook
ultimo_modulo_real = modulos_historicos[-1] # '5126'
base_inicial = pd.read_excel(BASE_DIR / f'basealunos{ultimo_modulo_real}.xlsx')['Qtd_Ativos'].sum()

# Extraindo os totais simulados pelo modelo
modulos = [ultimo_modulo_real, '5226', '5326', '5426']
valores = [
    base_inicial,
    df_projecao['Base_Final_5226'].sum(),
    df_projecao['Base_Final_5326'].sum(),
    df_projecao['Base_Final_5426'].sum()
]

# Formatando os nomes dos eixos para o padrão do seu painel (ex: '2026-51')
labels = [f"20{m[2:]}-{m[:2]}" for m in modulos]

# 3. Criação da Figura
plt.figure(figsize=(12, 6))

# Plotagem das barras verticais (Azul institucional)
ax = sns.barplot(x=labels, y=valores, color='#005b96')

# Adicionando uma linha de tendência (Laranja) para destacar a velocidade da queda
plt.plot(labels, valores, color='#ff6f00', marker='o', linewidth=2.5, markersize=8)

# 4. Formatação Executiva (Títulos, Rótulos e Limpeza)
plt.title('Evolução Acumulada da Base de Alunos | Instituicao EAD', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Calendário Acadêmico', fontsize=12, labelpad=10)
plt.ylabel('Qtd Alunos Ativos', fontsize=12)

# Subindo o limite do gráfico para os números não cortarem
plt.ylim(0, max(valores) * 1.15) 

# Escrevendo os valores exatos no topo de cada barra
for i, v in enumerate(valores):
    # Se for o módulo real (primeiro), adiciona um (Real) na frente
    texto_valor = f"{v:,.0f}" if i > 0 else f"{v:,.0f}\n(Real)"
    ax.text(i, v + max(valores)*0.03, texto_valor, color='black', ha='center', fontweight='bold', fontsize=11)

# Limpando bordas e o eixo Y (já que os números estão nas barras)
sns.despine(left=True)
ax.set_yticks([]) 

# Renderiza o gráfico
plt.tight_layout()
plt.show()